# IDC Consolidated Report -- Indicators 5, 6, 7 + Participants

Builds the first half of `IDC Report Consolidated Y2.xlsx`: Indicator 5 (passed
through from the database extract), Indicator 7 (all support service events:
template submissions + database-logged services), Indicator 6 (one row per
supported learner, derived from 7), and the Support Participants List.

## Run order -- this matters

1. `support_services_consolidated.ipynb` -- produces `indicator_7_data.csv`
2. **This notebook** -- CREATES the consolidated workbook (`mode='w'`) with the
   four sheets above
3. `eos_refactored.ipynb` -- APPENDS the earning opportunities sheets
   (`mode='a'`, `if_sheet_exists='replace'`)

Running 3 before 2 is safe only if the workbook already exists from a prior
run of 2; running 2 after 3 wipes the EO sheets. When in doubt: 1, 2, 3.

## Monthly usage

Edit `MONTH_DIR` in CONFIG (and `REPORT_START` once a year), Restart & Run All.
If the notebook halts, the validation table at the failure point names what is
wrong and where to fix it.

## A note on `cohort_name`

The support services CSV carries a `cohort_name` column. It was added to the
non-financial data points by request a few reporting cycles ago and the request
was later reversed -- but the column stays useful *upstream* (cohort mapping,
validations, audit of the standalone CSV), so it is kept there and dropped
here, at the consolidation boundary. None of the consolidated sheets include
cohort.

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

## 1. Configuration -- the only section to edit month to month

In [ ]:
# Y2 reporting window start. Services dated before this are dropped (with a report).
REPORT_START = pd.Timestamp('2026-04-01')

BASE = '/Users/markngendo/Documents/Umuzi/Reporting'
MONTH_DIR = f'{BASE}/Monthly IDC/June (2026)'   # <-- the monthly edit

PATHS = {
    'rich':                f'{BASE}/Improved IDC/Categories/Support/db_fields.csv',
    'indicator_5_db':      f'{MONTH_DIR}/Database/indicator_5.csv',
    'indicator_7_db':      f'{MONTH_DIR}/Database/indicator_7.csv',
    'indicator_7_support': f'{MONTH_DIR}/Sink Datasets/indicator_7_data.csv',
    # Set to a prior month's indicator_7_data.csv if a data owner submits
    # non-cumulatively and last month's template rows need to ride along.
    # Leave as None in a normal month.
    'prev_indicator_7':    None,
    'indicator_6_out':     f'{MONTH_DIR}/Sink Datasets/indicator_6_data.csv',
    'report_out':          f'{MONTH_DIR}/Sink Datasets/IDC Report Consolidated Y2.xlsx',
}

SHEET_NAMES = {
    'five':         'Indicator 5(# Registrations)',
    'six':          'Indicator 6(# People Supported)',
    'seven':        'Indicator 7(# of Services)',
    # the old notebook wrote 'Support Paricipants List' (typo). Corrected here;
    'participants': 'Support Participants List',
}

# Column contract for indicator 7 inputs (support CSV also carries cohort_name,
# which is dropped at ingest -- see the note above).
IND7_COLS = [
    'learner_id', 'application_id', 'umuzi_email',
    'first_name', 'last_name', 'cellphone_number',
    'id_number', 'gender', 'date_of_birth', 'age_service_accessed', 'age_range', 'race',
    'residential_area_type', 'has_disability_or_differently_abled',
    'application_date', 'nearest_metro', 'province', 'date_service_accessed',
    'month_of_service_accessed', 'service_used', 'service_name',
]

## 2. Helpers and validation framework

Same `Checks` framework as `support_services_consolidated`: failures print their
diagnostic and offending rows immediately; `halt_if_failed()` stops the run with
the full picture in one table.

In [ ]:
def parse_date(val):
    """Parse the date formats that appear across the IDC files."""
    formats = [
        "%Y-%m-%d %H:%M:%S.%f",
        "%Y-%m-%d %H:%M:%S",
        "%Y-%m-%d",
        "%d/%m/%Y %H:%M:%S.%f",
        "%d/%m/%Y %H:%M:%S",
        "%d/%m/%Y",
    ]
    for fmt in formats:
        try:
            return pd.to_datetime(val, format=fmt)
        except Exception:
            continue
    try:
        return pd.to_datetime(val)
    except Exception:
        return pd.NaT


def normalize_emails(series):
    return series.str.strip().str.lower()


class Checks:
    """Collects validation results and prints failures with actionable detail."""

    def __init__(self):
        self.rows = []

    def add(self, stage, name, passed, detail="", offenders=None, level="error"):
        self.rows.append({
            "stage": stage, "check": name,
            "status": "PASS" if passed else ("FAIL" if level == "error" else "WARN"),
            "level": level, "detail": "" if passed else detail,
        })
        if not passed:
            tag = "FAIL" if level == "error" else "WARN"
            print(f"[{tag}] {stage} :: {name}")
            if detail:
                print(f"       {detail}")
            if offenders is not None and len(offenders):
                display(offenders)

    def summary(self):
        return pd.DataFrame(self.rows)

    def halt_if_failed(self, stage=None):
        df = self.summary()
        if stage is not None:
            df = df[df["stage"] == stage]
        failed = df[df["status"] == "FAIL"]
        if not failed.empty:
            display(failed[["stage", "check", "detail"]])
            raise AssertionError(
                f"{len(failed)} validation failure(s). Review the table above; "
                "each detail line names the file or CONFIG entry to fix. Then "
                "Restart & Run All."
            )


checks = Checks()

## 3. Enrichment data (`db_fields.csv`)

Both email columns are normalized (the old notebook lowercased only the *values*
of the personal-to-umuzi map, not its index, so any personal email stored with
stray case in the database could never match -- fixed here). Deduplication keeps
the highest `learner_id` per umuzi email, matching the EOs notebook.

In [ ]:
rich = pd.read_csv(PATHS['rich'], dtype={'id_number': str, 'cellphone_number': str})

rich['umuzi_email'] = normalize_emails(rich['umuzi_email'])
rich['email'] = normalize_emails(rich['email'])

rich = rich.sort_values('learner_id').drop_duplicates(subset='umuzi_email', keep='last')
assert rich['umuzi_email'].is_unique

dup_personal = rich.loc[rich['email'].notna() & rich.duplicated(subset='email', keep=False)]
checks.add('rich', 'personal email maps to a single umuzi_email', dup_personal.empty,
           'The same personal email maps to multiple umuzi emails; keeping the '
           'highest learner_id for the remap.',
           offenders=dup_personal[['email', 'umuzi_email', 'learner_id']].sort_values('email'),
           level='warn')

emailMap = (rich.dropna(subset=['email'])
            .drop_duplicates(subset='email', keep='last')
            .set_index('email')['umuzi_email'])

print(f"db_fields loaded: {rich.shape[0]} learners, {emailMap.shape[0]} personal-email remaps")

## 4. Indicator 5 (database extract, passed through)

Written to its sheet untouched -- checks here are structural only, so a broken
export is caught before it reaches the partner workbook.

In [ ]:
five = pd.read_csv(PATHS['indicator_5_db'],
                   dtype={'id_number': 'str', 'cellphone_number': 'str'})

checks.add('ind5', 'indicator 5 extract contains rows', len(five) > 0,
           'indicator_5.csv is empty. Re-run idc_indicator_5.sql and re-export.')

n_dup5 = int(five.duplicated().sum())
checks.add('ind5', 'indicator 5 has no duplicate rows', n_dup5 == 0,
           f'{n_dup5} duplicate row(s) in the extract. They are passed through '
           'as-is -- check the query before submitting.', level='warn')

print(f"indicator 5: {five.shape[0]} rows x {five.shape[1]} cols")

## 5. Indicator 7 inputs (support CSV + database extract)

Each input is checked against the column contract before the concat -- a silent
mismatch here is what NaN-fills half a sheet. `cohort_name` is dropped from the
support CSV at ingest (see the note at the top). A `source` column tags every
row for diagnostics and is removed at export.

In [ ]:
def load_ind7(path, name, drop_cohort=False):
    df = pd.read_csv(path, dtype={'id_number': 'str', 'cellphone_number': 'str'})
    if drop_cohort and 'cohort_name' in df.columns:
        df = df.drop(columns='cohort_name')

    missing = set(IND7_COLS) - set(df.columns)
    extra = set(df.columns) - set(IND7_COLS)
    checks.add('load', f'{name}: required columns present', not missing,
               f'Missing: {sorted(missing)}. The file does not match the '
               'indicator 7 contract -- do not concat until fixed.')
    checks.add('load', f'{name}: no unexpected columns', not extra,
               f'Unexpected: {sorted(extra)}. They will be dropped.', level='warn')
    checks.add('load', f'{name}: file contains rows', len(df) > 0,
               'Loaded 0 rows -- check the path in CONFIG.', level='warn')

    df = df[[c for c in IND7_COLS if c in df.columns]].copy()
    df['source'] = name
    print(f"[ok] {name}: {len(df)} rows")
    return df


parts = [
    load_ind7(PATHS['indicator_7_support'], 'support', drop_cohort=True),
    load_ind7(PATHS['indicator_7_db'], 'database'),
]
if PATHS['prev_indicator_7']:
    parts.insert(0, load_ind7(PATHS['prev_indicator_7'], 'prev', drop_cohort=True))

checks.halt_if_failed('load')

combined = pd.concat(parts, ignore_index=True)
print(f"\nCombined: {combined.shape[0]} rows from {len(parts)} input(s)")

## 6. Canonical emails, dates, and the reporting window

In [ ]:
combined['umuzi_email'] = normalize_emails(combined['umuzi_email'])
combined['umuzi_email'] = combined['umuzi_email'].map(lambda x: emailMap.get(x, x))

# parse dates immediately after read -- CSV strings and any prev-file mixing
# must never reach a concat or a .str operation as mixed types
combined['date_of_birth'] = combined['date_of_birth'].map(parse_date)
combined['date_service_accessed'] = combined['date_service_accessed'].map(parse_date)

no_date = combined['date_service_accessed'].isna()
if no_date.any():
    print(f"{no_date.sum()} row(s) without a parseable service date -> dropped:")
    print(combined.loc[no_date, ['source', 'umuzi_email', 'service_name']].to_string())
combined = combined.loc[~no_date]

pre_window = combined['date_service_accessed'] < REPORT_START
if pre_window.any():
    print(f"{pre_window.sum()} row(s) before {REPORT_START.date()} (outside Y2) -> dropped")
    display(combined.loc[pre_window]
            .groupby(['source', combined.loc[pre_window, 'date_service_accessed']
                      .dt.to_period('M').rename('month')])
            .size().rename('rows').reset_index())
combined = combined.loc[~pre_window]

future = combined['date_service_accessed'] > pd.Timestamp('now')
checks.add('dates', 'no service dates in the future', not future.any(),
           'Future dates usually mean a day/month swap at source:',
           offenders=combined.loc[future, ['source', 'umuzi_email', 'date_service_accessed']])
checks.halt_if_failed('dates')

# Month name only (no year) on the consolidated sheets -- this is the partner
# format the old notebook produced, and the Y2 window makes it unambiguous.
# The support CSV's 'Month YYYY' values are deliberately overwritten.
combined['month_of_service_accessed'] = combined['date_service_accessed'].dt.month_name()
combined = combined.reset_index(drop=True)
combined.shape

## 7. Re-enrichment of nameless rows

Rows arriving with no first *and* no last name (typically database-extract
records whose demographic fields are incomplete) get their service columns kept
and their demographics re-merged from `db_fields`. Ages are recomputed where the
re-merge supplied a date of birth. Anything still nameless afterwards is
reported per source -- those learners genuinely need fixing in the database.

In [ ]:
nameless = combined['first_name'].isna() & combined['last_name'].isna()

if nameless.any():
    print(f"{nameless.sum()} nameless row(s) found -- re-enriching from db_fields:")
    print(combined.loc[nameless].groupby('source').size().rename('rows').to_string())

    keep_cols = ['umuzi_email', 'service_used', 'service_name', 'date_service_accessed',
                 'age_service_accessed', 'month_of_service_accessed', 'age_range', 'source']
    enriched = (combined.loc[nameless, keep_cols]
                .merge(rich.drop(columns=['email']), on='umuzi_email', how='left'))

    combined = (pd.concat([combined.loc[~nameless], enriched[combined.columns]],
                          ignore_index=True)
                .sort_values('umuzi_email').reset_index(drop=True))

    combined['date_of_birth'] = combined['date_of_birth'].map(parse_date)

    # recompute ages where the re-merge supplied a date of birth
    needs_age = combined['age_service_accessed'].isna() & combined['date_of_birth'].notna()
    if needs_age.any():
        combined.loc[needs_age, 'age_service_accessed'] = (
            (combined.loc[needs_age, 'date_service_accessed']
             - combined.loc[needs_age, 'date_of_birth']).dt.days // 365
        )
        combined.loc[needs_age, 'age_range'] = pd.cut(
            combined.loc[needs_age, 'age_service_accessed'],
            bins=[-np.inf, 17, 25, 35, np.inf],
            labels=['17 and below', '18-25', '26-35', '36+'])
        print(f"recomputed age for {needs_age.sum()} re-enriched row(s)")

still_nameless = combined.loc[combined['first_name'].isna() & combined['last_name'].isna()]
checks.add('enrich', 'every row has a learner name', still_nameless.empty,
           'These learners are missing from db_fields (or the email cannot be '
           'canonicalized). Fix in the database and re-export db_fields.csv; '
           'they ship nameless until then:',
           offenders=(still_nameless.groupby(['source', 'umuzi_email'])
                      .size().rename('rows').reset_index()),
           level='warn')

## 8. Deduplication

Exact duplicates are dropped with a per-source report. Near-duplicates (same
learner, service, and date arriving from *different* inputs) are warned about
but kept -- a template submission and a database log of the same session is a
data-owner conversation, not something to silently resolve here.

In [ ]:
exact = combined.duplicated()
if exact.any():
    print(f"{exact.sum()} exact duplicate row(s) -> dropped:")
    print(combined.loc[exact].groupby('source').size().rename('rows').to_string())
combined = combined.loc[~exact].reset_index(drop=True)

near_key = ['umuzi_email', 'service_used', 'service_name', 'date_service_accessed']
near = combined.loc[combined.duplicated(subset=near_key, keep=False)]
cross_source = near.groupby(near_key, observed=True)['source'].nunique()
n_cross = int((cross_source > 1).sum())
checks.add('dedup', 'no same service event reported by multiple inputs', n_cross == 0,
           f'{n_cross} learner/service/date combination(s) appear in more than one '
           'input (e.g. both a template and the database). Kept as-is -- verify '
           'with the data owners whether these are double-reported:',
           offenders=near.sort_values(near_key)[['source'] + near_key].head(20),
           level='warn')

## 9. Indicator 6 (one row per supported learner)

Derived from the cleaned `combined`. Two fixes over the old notebook: the
service list is sorted (deterministic across runs), and the month now comes
from the learner's **latest service date** -- the old `max` on month *names*
was alphabetical, so a learner served in May and June reported month "May"
(M > J) alongside a June date.

In [ ]:
ind6 = (combined
        .groupby('umuzi_email', as_index=False)
        .agg({
            'service_used': 'first',
            'service_name': lambda x: sorted(set(x)),
            'learner_id': 'first',
            'application_id': 'first',
            'application_date': 'first',
            'first_name': 'first',
            'last_name': 'first',
            'cellphone_number': 'first',
            'id_number': 'first',
            'race': 'first',
            'gender': 'first',
            'date_of_birth': 'first',
            'age_service_accessed': 'first',
            'age_range': 'first',
            'has_disability_or_differently_abled': 'first',
            'nearest_metro': 'first',
            'province': 'first',
            'residential_area_type': 'first',
            'date_service_accessed': 'max',
        }))

# month derived from the max date, not an alphabetical max of month names
ind6['month_of_service_accessed'] = ind6['date_service_accessed'].dt.month_name()

ind6 = ind6[IND7_COLS]

checks.add('ind6', 'one row per supported learner',
           len(ind6) == combined['umuzi_email'].nunique(),
           f"ind6 has {len(ind6)} rows but combined has "
           f"{combined['umuzi_email'].nunique()} unique learners.")
checks.halt_if_failed('ind6')

ind6.to_csv(PATHS['indicator_6_out'], index=False)
print(f"indicator 6: {len(ind6)} supported learner(s) -> {PATHS['indicator_6_out']}")

## 10. Support Participants List

In [ ]:
participantsCols = [
    'first_name', 'last_name', 'id_number', 'date_service_accessed', 'service_name',
    'cellphone_number', 'age_service_accessed', 'nearest_metro', 'province', 'race', 'gender',
]
participantsList = combined[participantsCols].copy()

n_nameless = int(participantsList['first_name'].isna().sum())
checks.add('participants', 'participants list has no nameless rows', n_nameless == 0,
           f'{n_nameless} row(s) without a first name (see the enrichment report '
           'above for the emails involved).', level='warn')
print(f"participants list: {len(participantsList)} rows")

## 11. Final validation summary and export

The workbook is created fresh (`mode='w'`) -- the EOs notebook appends to it
afterwards. The `source` diagnostic column is dropped from the Indicator 7
sheet here, at export only.

In [ ]:
bad_id = combined.loc[combined['id_number'].notna() & (combined['id_number'].str.len() != 13)]
checks.add('final', 'id_number is 13 digits', bad_id.empty,
           'SA ID numbers should be 13 digits:',
           offenders=bad_id[['source', 'umuzi_email', 'id_number']].drop_duplicates(),
           level='warn')

print("Rows per source and month:")
display(combined.groupby(['source', 'month_of_service_accessed'])
        .size().rename('rows').reset_index()
        .pivot(index='source', columns='month_of_service_accessed', values='rows')
        .fillna(0).astype(int))

print("\nFull validation summary:")
display(checks.summary())
checks.halt_if_failed()

In [ ]:
with pd.ExcelWriter(PATHS['report_out'], mode='w') as writer:
    five.to_excel(writer, sheet_name=SHEET_NAMES['five'], index=False)
    ind6.to_excel(writer, sheet_name=SHEET_NAMES['six'], index=False)
    combined.drop(columns='source').to_excel(writer, sheet_name=SHEET_NAMES['seven'], index=False)
    participantsList.to_excel(writer, sheet_name=SHEET_NAMES['participants'], index=False)

print(f"Wrote {PATHS['report_out']}:")
print(f"  {SHEET_NAMES['five']:30s} {five.shape[0]:>6} rows")
print(f"  {SHEET_NAMES['six']:30s} {ind6.shape[0]:>6} rows")
print(f"  {SHEET_NAMES['seven']:30s} {combined.shape[0]:>6} rows")
print(f"  {SHEET_NAMES['participants']:30s} {participantsList.shape[0]:>6} rows")
print("\nNEXT STEP: run eos_refactored.ipynb to append the earning opportunities "
      "sheets to this workbook.")